# Mesh extraction correctness check — resolution convergence (scan105)

Self-contained, single-cell. Loads the trained SDF checkpoint fresh from disk
and re-runs Marching Cubes at several grid resolutions (not just the pipeline's
default 160^3), checking `watertight` / `euler_number` / `genus` / boundary
edges at each.

**Why this matters:** the pipeline's own diagnostic (cell 40 in
`pipeline_run_all_fix_3`) already found watertight=True, 0 boundary edges,
genus=1 at resolution 160. That rules out a hole-filling / post-process bug
(nothing to fill when there are 0 boundary edges). But it does NOT yet rule
out a *discretization* artifact -- i.e. the handle could in principle be an
aliasing effect of a 160^3 grid that a finer grid would resolve away. This
notebook closes that last gap: if genus stays 1 as resolution increases, the
handle is confirmed to be real learned SDF geometry, not a grid artifact --
extraction stage is then verified 100% correct and the fix belongs entirely
on the training side (SDF / normal-consistency), not here.

In [ ]:
import subprocess, sys as _sys
try:
    import trimesh
except ImportError:
    subprocess.run([_sys.executable, "-m", "pip", "install", "--quiet", "trimesh"], check=True)
    import trimesh

import sys
import json
from pathlib import Path
import numpy as np
import torch
from trimesh.grouping import group_rows

# ---- 0. Locate project + most recent scan105 run (same convention as the
#         other standalone diagnostic notebooks in this project) ----
def _find_project_root(marker="src/utils.py"):
    for root in (Path("/notebooks"), Path("/content"), Path("/kaggle/working"), Path.home(), Path.cwd()):
        if root.exists():
            for c in root.glob("**/SDF-GS-Refine-implementation"):
                if (c / marker).exists():
                    return c
    return None

PROJECT_ROOT = _find_project_root()
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate SDF-GS-Refine-implementation project root.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import sdf as sdf_mod
from src import mesh as mesh_mod
from src.utils import load_config

CFG = load_config(str(PROJECT_ROOT / "configs" / "config.json"))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def _torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

run_dir = sorted(PROJECT_ROOT.glob("run_artifacts/scan105-*/02_training_checkpoint.pt"),
                  key=lambda p: p.stat().st_mtime, reverse=True)[0].parent
print(f"[OK] Run dir: {run_dir}")

ckpt = _torch_load(run_dir / "02_training_checkpoint.pt")

sdf_cfg = CFG["sdf_network"]
sdf_net = sdf_mod.SDFNetwork(d_in=3, d_hidden=sdf_cfg["d_hidden"], n_layers=sdf_cfg["n_layers"],
                              radius_init=sdf_cfg["radius_init"], beta=sdf_cfg["softplus_beta"]).to(DEVICE)
sdf_net.load_state_dict(ckpt["sdf_state"])
sdf_net.eval()

# ---- Bounds: NO ply reload. The real scan105 .ply is confirmed gone from
# disk (run_identity.json's recorded path resolves to exists()=False, and a
# full-disk search turned up zero candidates -- verified interactively before
# writing this). Reloading ANY other point_cloud.ply found on disk (e.g. a
# synthetic_sphere stand-in) silently produces the WRONG bounding box for
# THIS checkpoint -- that's exactly the bug this cell replaces.
#
# Instead: derive bounds from 03_raw_mesh.npz -- the RAW marching-cubes
# output the ORIGINAL pipeline run itself already saved to disk, back when
# the real .ply still existed and bounds were computed correctly (cell 13 of
# pipeline_run_all_fix_3). Those vertices ARE real scan105 teddy-bear
# geometry -- no reload, no download, no ambiguity about which object.
_raw_path = run_dir / "03_raw_mesh.npz"
if not _raw_path.exists():
    raise FileNotFoundError(
        f"{_raw_path} not found either -- there is no way left to recover real scan105 "
        f"bounds without either that file or the original .ply. At this point the only "
        f"option is a real Kaggle re-download of the scan105 dataset -- confirm with the "
        f"user before doing that, don't do it silently."
)
_raw = np.load(_raw_path)
_verts0 = _raw["verts"]
_bmin = _verts0.min(axis=0)
_bmax = _verts0.max(axis=0)
_extent = _bmax - _bmin

# Generous safety margin: the original bounds were computed from the
# Gaussian bbox (+10% margin), which is a SUPERSET of the resulting mesh's
# vertex bbox (mesh vertices sit inside that box, not exactly on its edges).
# So this reconstruction under-estimates the true original box slightly --
# pad generously (25%) so re-extraction at higher resolutions can't clip
# anything that genuinely belongs inside.
_SAFETY_MARGIN = 0.25
bounds_min = _bmin - _SAFETY_MARGIN * _extent
bounds_max = _bmax + _SAFETY_MARGIN * _extent
print(f"[BOUNDS] (reconstructed from real 03_raw_mesh.npz, {len(_verts0)} verts, "
      f"safety_margin={_SAFETY_MARGIN}) min={bounds_min} max={bounds_max}")
print(f"[SANITY] original run (pipeline cell 29) reported raw mesh V=47680, F=95360 at "
      f"res=160 -- this run's res=160 result below should land in that neighborhood "
      f"if bounds are sane; a wildly different count means something's still off.")

TUNNEL_CENTROID = np.array([0.1353, -0.1619, -0.0745])  # from cell 42's independent geodesic-ratio localization

# ---- 1. Re-run Marching Cubes at several resolutions, check topology at each ----
RESOLUTIONS = [160, 224, 288]  # 160 = pipeline baseline (reproduces the known genus=1 result)

results = []
for res in RESOLUTIONS:
    verts, faces = mesh_mod.extract_mesh(sdf_net, bounds_min, bounds_max, res, DEVICE)
    if verts is None:
        print(f"[res={res}] Marching Cubes found NO zero-crossing -- unexpected, flag this.")
        results.append(dict(res=res, ok=False))
        continue

    m = trimesh.Trimesh(vertices=verts, faces=faces, process=False)
    watertight = bool(m.is_watertight)
    euler = int(m.euler_number)
    boundary_groups = group_rows(m.edges_sorted, require_count=1)
    n_boundary = len(boundary_groups)
    genus = (2 - euler) // 2 if watertight else None

    # distance from mesh vertices to the known handle centroid, to confirm
    # the handle (if present) is still sitting in the same place, not just
    # coincidentally genus=1 somewhere else
    d_to_centroid = np.linalg.norm(verts - TUNNEL_CENTROID, axis=1).min()

    print(f"[res={res:>3d}] V={len(verts):6d} F={len(faces):6d}  watertight={watertight!s:5s}  "
          f"euler={euler:+d}  boundary_edges={n_boundary}  genus={genus}  "
          f"nearest_vertex_to_known_handle={d_to_centroid:.4f}")
    results.append(dict(res=res, ok=True, watertight=watertight, euler=euler,
                         n_boundary=n_boundary, genus=genus))

# ---- 2. Verdict ----
print("\n=== MESH EXTRACTION CORRECTNESS VERDICT ===")
ok_results = [r for r in results if r["ok"]]
all_watertight = all(r["watertight"] for r in ok_results)
all_zero_boundary = all(r["n_boundary"] == 0 for r in ok_results)
genus_values = {r["genus"] for r in ok_results if r["watertight"]}

if len(ok_results) < len(RESOLUTIONS):
    print(">>> INCOMPLETE: Marching Cubes failed to find a zero-crossing at one or more "
          "resolutions -- extraction is NOT verified correct, investigate bounds/isovalue first.")
elif not all_watertight:
    print(">>> NOT CLEAN: mesh is not watertight at every resolution tested -- there IS a real "
          "extraction/post-process gap issue, separate from the training-side normal-conflict finding. "
          "Investigate marching_cubes boundary handling before touching training.")
elif not all_zero_boundary:
    print(">>> NOT CLEAN: boundary edges present at some resolution -- extraction has an open-mesh "
          "bug independent of the genus-1 handle. Investigate before touching training.")
elif genus_values == {1}:
    print(f">>> CONFIRMED 100%: mesh extraction is CORRECT. Watertight and 0 boundary edges at every "
          f"resolution tested ({RESOLUTIONS}), and genus stays exactly 1 at every resolution -- the "
          f"handle is resolution-independent, i.e. it is real geometry baked into the trained SDF's "
          f"zero-level-set, NOT a 160^3 grid discretization artifact. Marching Cubes and postprocess() "
          f"are both doing exactly what they should. The fix belongs entirely on the training side "
          f"(normal-consistency at the tunnel region), not in the extraction stage -- do not spend any "
          f"more time auditing Stage D/E.")
else:
    print(f">>> UNSTABLE: genus varies across resolutions ({sorted(genus_values)}) -- this DOES point "
          f"to a discretization-sensitive region (grid too coarse somewhere), which is an extraction-"
          f"side concern after all. Bump grid_resolution in the real pipeline run and re-check, before "
          f"assuming the training-side fix alone will resolve it.")
